In [3]:
import seaborn as sns
import pandas as pd
import polars as pl
import numpy as np

Анализ данных с помощью объединений и конкатенации

Набор данных: Используйте набор данных "tips" и набор данных "flights" из Seaborn.

Инструкции:

Загрузите наборы данных "tips" и "flights" в отдельные Polars DataFrames.

Из набора данных 'flights' выберите столбцы 'year', 'month' и 'passengers'.

Создайте новый DataFrame из 'flights', который агрегирует общее количество пассажиров за год.

Объедините эти агрегированные данные с набором данных 'tips' по столбцу 'year' (примечание: для этого вам может понадобиться создать столбец 'year' в 'tips').

Конкатенируйте DataFrame 'tips' (отфильтрованный по времени 'Dinner') с первыми 50 строками 'flights' (подумайте, каким образом).

Отобразите первые 10 строк объединенного и конкатенированного DataFrame.

In [4]:
# Анализ данных с помощью объединений и конкатенации
# Набор данных: Используйте набор данных "tips" и набор данных "flights" из Seaborn.
tips = sns.load_dataset('tips')
flights = sns.load_dataset('flights')


In [5]:
# Инструкции:
# Загрузите наборы данных "tips" и "flights" в отдельные Polars DataFrames.
df_tips = pl.from_pandas(tips)
display(df_tips)
df_flights = pl.from_pandas(flights)
display(df_flights)

total_bill,tip,sex,smoker,day,time,size
f64,f64,cat,cat,cat,cat,i64
16.99,1.01,"""Female""","""No""","""Sun""","""Dinner""",2
10.34,1.66,"""Male""","""No""","""Sun""","""Dinner""",3
21.01,3.5,"""Male""","""No""","""Sun""","""Dinner""",3
23.68,3.31,"""Male""","""No""","""Sun""","""Dinner""",2
24.59,3.61,"""Female""","""No""","""Sun""","""Dinner""",4
…,…,…,…,…,…,…
29.03,5.92,"""Male""","""No""","""Sat""","""Dinner""",3
27.18,2.0,"""Female""","""Yes""","""Sat""","""Dinner""",2
22.67,2.0,"""Male""","""Yes""","""Sat""","""Dinner""",2


year,month,passengers
i64,cat,i64
1949,"""Jan""",112
1949,"""Feb""",118
1949,"""Mar""",132
1949,"""Apr""",129
1949,"""May""",121
…,…,…
1960,"""Aug""",606
1960,"""Sep""",508
1960,"""Oct""",461


In [6]:
# Из набора данных 'flights' выберите столбцы 'year', 'month' и 'passengers'.
df_flights_selected = df_flights.select(['year', 'month', 'passengers'])
df_flights_selected.head(3)

year,month,passengers
i64,cat,i64
1949,"""Jan""",112
1949,"""Feb""",118
1949,"""Mar""",132


In [7]:
# Создайте новый DataFrame из 'flights', который агрегирует общее количество пассажиров за год.
df_flights_agg = df_flights.group_by('year').agg(pl.col('passengers').sum().alias('total_passengers'))
display(df_flights_agg.head(3))


year,total_passengers
i64,i64
1960,5714
1958,4572
1950,1676


In [8]:
# Объедините эти агрегированные данные с набором данных 'tips' по столбцу 'year' 
# (примечание: для этого вам может понадобиться создать столбец 'year' в 'tips').

# Единственная идея это создать список годов из flights а потом рандомно распределить по все 244 строкам

min_year = df_flights_agg.select(pl.col("year").min()).item()
print(f"Минимальный год : {min_year}")
n_years = df_flights_agg.select(pl.col("year").n_unique()).item()
print(f"Количество уникальных лет: {n_years}")
max_year = df_flights_agg.select(pl.col("year").max()).item()
print(f"Максимальный год : {max_year}")
years_array = list(range(min_year, max_year + 1))
print(f"Массив лет: {years_array}")


years_repeated = np.random.choice(years_array, size=len(df_tips))


df_tips_with_year = df_tips.with_columns(pl.Series("year", years_repeated))
display(df_tips_with_year.head(10))

df_merged = df_flights_agg.join(df_tips_with_year, on='year', how='left')
display(df_merged)

Минимальный год : 1949
Количество уникальных лет: 12
Максимальный год : 1960
Массив лет: [1949, 1950, 1951, 1952, 1953, 1954, 1955, 1956, 1957, 1958, 1959, 1960]


total_bill,tip,sex,smoker,day,time,size,year
f64,f64,cat,cat,cat,cat,i64,i64
16.99,1.01,"""Female""","""No""","""Sun""","""Dinner""",2,1952
10.34,1.66,"""Male""","""No""","""Sun""","""Dinner""",3,1950
21.01,3.5,"""Male""","""No""","""Sun""","""Dinner""",3,1958
23.68,3.31,"""Male""","""No""","""Sun""","""Dinner""",2,1956
24.59,3.61,"""Female""","""No""","""Sun""","""Dinner""",4,1951
25.29,4.71,"""Male""","""No""","""Sun""","""Dinner""",4,1950
8.77,2.0,"""Male""","""No""","""Sun""","""Dinner""",2,1953
26.88,3.12,"""Male""","""No""","""Sun""","""Dinner""",4,1950
15.04,1.96,"""Male""","""No""","""Sun""","""Dinner""",2,1951


year,total_passengers,total_bill,tip,sex,smoker,day,time,size
i64,i64,f64,f64,cat,cat,cat,cat,i64
1960,5714,14.83,3.02,"""Female""","""No""","""Sun""","""Dinner""",2
1960,5714,20.29,2.75,"""Female""","""No""","""Sat""","""Dinner""",2
1960,5714,9.55,1.45,"""Male""","""No""","""Sat""","""Dinner""",2
1960,5714,16.31,2.0,"""Male""","""No""","""Sat""","""Dinner""",3
1960,5714,13.94,3.06,"""Male""","""No""","""Sun""","""Dinner""",2
…,…,…,…,…,…,…,…,…
1949,1520,8.58,1.92,"""Male""","""Yes""","""Fri""","""Lunch""",1
1949,1520,20.45,3.0,"""Male""","""No""","""Sat""","""Dinner""",4
1949,1520,15.53,3.0,"""Male""","""Yes""","""Sat""","""Dinner""",2


In [13]:
# Конкатенируйте DataFrame 'tips' (отфильтрованный по времени 'Dinner') с первыми 50 строками 'flights' (подумайте, каким образом).
df_tips_dinner = df_tips.filter(pl.col('time') == 'Dinner')
display(df_tips_dinner)
df_flights_head = df_flights.head(50)
display(df_flights_head)
df_concatenated = pl.concat([df_tips_dinner, df_flights_head], how='diagonal_relaxed')
display(df_concatenated)


total_bill,tip,sex,smoker,day,time,size
f64,f64,cat,cat,cat,cat,i64
16.99,1.01,"""Female""","""No""","""Sun""","""Dinner""",2
10.34,1.66,"""Male""","""No""","""Sun""","""Dinner""",3
21.01,3.5,"""Male""","""No""","""Sun""","""Dinner""",3
23.68,3.31,"""Male""","""No""","""Sun""","""Dinner""",2
24.59,3.61,"""Female""","""No""","""Sun""","""Dinner""",4
…,…,…,…,…,…,…
29.03,5.92,"""Male""","""No""","""Sat""","""Dinner""",3
27.18,2.0,"""Female""","""Yes""","""Sat""","""Dinner""",2
22.67,2.0,"""Male""","""Yes""","""Sat""","""Dinner""",2


year,month,passengers
i64,cat,i64
1949,"""Jan""",112
1949,"""Feb""",118
1949,"""Mar""",132
1949,"""Apr""",129
1949,"""May""",121
…,…,…
1952,"""Oct""",191
1952,"""Nov""",172
1952,"""Dec""",194


total_bill,tip,sex,smoker,day,time,size,year,month,passengers
f64,f64,cat,cat,cat,cat,i64,i64,cat,i64
16.99,1.01,"""Female""","""No""","""Sun""","""Dinner""",2,null,null,null
10.34,1.66,"""Male""","""No""","""Sun""","""Dinner""",3,null,null,null
21.01,3.5,"""Male""","""No""","""Sun""","""Dinner""",3,null,null,null
23.68,3.31,"""Male""","""No""","""Sun""","""Dinner""",2,null,null,null
24.59,3.61,"""Female""","""No""","""Sun""","""Dinner""",4,null,null,null
…,…,…,…,…,…,…,…,…,…
null,null,null,null,null,null,null,1952,"""Oct""",191
null,null,null,null,null,null,null,1952,"""Nov""",172
null,null,null,null,null,null,null,1952,"""Dec""",194


In [15]:
# Отобразите первые 10 строк объединенного и конкатенированного DataFrame.
display(df_concatenated.head(10))
display(df_concatenated.tail(10))

total_bill,tip,sex,smoker,day,time,size,year,month,passengers
f64,f64,cat,cat,cat,cat,i64,i64,cat,i64
16.99,1.01,"""Female""","""No""","""Sun""","""Dinner""",2,null,null,null
10.34,1.66,"""Male""","""No""","""Sun""","""Dinner""",3,null,null,null
21.01,3.5,"""Male""","""No""","""Sun""","""Dinner""",3,null,null,null
23.68,3.31,"""Male""","""No""","""Sun""","""Dinner""",2,null,null,null
24.59,3.61,"""Female""","""No""","""Sun""","""Dinner""",4,null,null,null
25.29,4.71,"""Male""","""No""","""Sun""","""Dinner""",4,null,null,null
8.77,2.0,"""Male""","""No""","""Sun""","""Dinner""",2,null,null,null
26.88,3.12,"""Male""","""No""","""Sun""","""Dinner""",4,null,null,null
15.04,1.96,"""Male""","""No""","""Sun""","""Dinner""",2,null,null,null


total_bill,tip,sex,smoker,day,time,size,year,month,passengers
f64,f64,cat,cat,cat,cat,i64,i64,cat,i64
null,null,null,null,null,null,null,1952,"""May""",183
null,null,null,null,null,null,null,1952,"""Jun""",218
null,null,null,null,null,null,null,1952,"""Jul""",230
null,null,null,null,null,null,null,1952,"""Aug""",242
null,null,null,null,null,null,null,1952,"""Sep""",209
null,null,null,null,null,null,null,1952,"""Oct""",191
null,null,null,null,null,null,null,1952,"""Nov""",172
null,null,null,null,null,null,null,1952,"""Dec""",194
null,null,null,null,null,null,null,1953,"""Jan""",196
